# Export the fine-tuned MagickHome Gemma to one GGUF file (no training)

Use this when you already have the trained adapter (`magick-gemma-lora.zip`, from the fine-tune notebook) and need
the **merged** model file for Ollama - e.g. because the fine-tune session disconnected during the export.
About 20-25 minutes on a free T4.

1. **Runtime -> Change runtime type -> T4 GPU.** 2. **Runtime -> Run all.** 3. Choose `magick-gemma-lora.zip` when asked.
4. The `.gguf` (~5 GB) downloads at the end. On the laptop: `bash finetune/install_local.sh ~/Downloads/<file>.gguf`
   -> Ollama model `magick-gemma` (the app gives it the short prompt automatically).

In [ ]:
QUANT = "q4_k_m"          # GGUF quantization (q4_k_m ~5 GB; q8_0 ~8 GB, a little better)
MAX_SEQ = 4096

In [ ]:
%%capture
!pip install -q unsloth

In [ ]:
import os, zipfile
from google.colab import files
print("Choose magick-gemma-lora.zip")
up = files.upload()
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall("lora_adapter")
print(sorted(os.listdir("lora_adapter")))

In [ ]:
from unsloth import FastModel
# loads the base model named in adapter_config.json and applies the adapter
model, tokenizer = FastModel.from_pretrained("lora_adapter", max_seq_length=MAX_SEQ, load_in_4bit=True)
model.save_pretrained_gguf("gguf_out", tokenizer, quantization_method=QUANT)
import glob
GGUF = sorted(glob.glob("gguf_out/*.gguf"), key=os.path.getsize)[-1]
print("Model file:", GGUF, f"({os.path.getsize(GGUF) / 1e9:.1f} GB)")

In [ ]:
files.download(GGUF)   # ~5 GB through the browser - keep this tab open until it finishes